# Recetas -> JSON (Gemini)

Este notebook convierte recetas desde `Recetas/` a JSON compatible con la webapp (`/recetas`) usando Gemini vía `src/LLMs_funcs.py`.

Qué hace por cada archivo:
- Extrae texto (HTML / DOCX / DOC / PDF / TXT).
- Llama a `recipe_text_to_webapp_json(...)`.
- Intenta extraer imagen si existe.
- Ajusta imagen a **360x270** y guarda en `images/recipes/`.
- Guarda JSON listo para subir en `data/recipes_generated/`.


In [1]:
from __future__ import annotations

import base64
import hashlib
import html
import io
import json
import os
import re
import subprocess
import tempfile
import time
import zipfile
from pathlib import Path
from typing import Any
from urllib.parse import unquote, urlparse

import requests
from PIL import Image, ImageOps, ImageStat
from dotenv import load_dotenv
from tqdm.auto import tqdm

import sys
import importlib.util

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / "secrets" / "env.prod")

if importlib.util.find_spec("google.genai") is None:
    raise RuntimeError("Missing `google-genai`. Install it with `pip install google-genai` in this environment.")

from src.LLMs_funcs import default_model, recipe_text_to_webapp_json

SOURCE_ROOT = PROJECT_ROOT / "Recetas"
OUTPUT_JSON_DIR = PROJECT_ROOT / "data" / "recipes_generated"
OUTPUT_IMAGE_DIR = PROJECT_ROOT / "images" / "recipes"

OUTPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_SOURCE_EXTENSIONS = {".html", ".htm", ".docx", ".doc", ".pdf", ".txt"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp"}

DEFAULT_CARD_COLOR = "rgb(118, 161, 146)"
MODEL = default_model
OUTPUT_LANGUAGE = "es"

if not (os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")):
    raise RuntimeError("No GEMINI_API_KEY / GOOGLE_API_KEY found. Check secrets/env.prod.")

print(f"Source root: {SOURCE_ROOT}")
print(f"Output JSON: {OUTPUT_JSON_DIR}")
print(f"Output images: {OUTPUT_IMAGE_DIR}")
print(f"Gemini model: {MODEL}")
print(f"Output language: {OUTPUT_LANGUAGE}")


/home/rafael/.conda/envs/csc_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Source root: /home/rafael/E/Python_Projects/Recetas_webapp/Recetas
Output JSON: /home/rafael/E/Python_Projects/Recetas_webapp/data/recipes_generated
Output images: /home/rafael/E/Python_Projects/Recetas_webapp/images/recipes
Gemini model: gemini-2.5-flash
Output language: es


In [2]:
def slugify(value: str) -> str:
    normalized = re.sub(r"[^a-z0-9]+", "-", str(value or "").strip().lower())
    return normalized.strip("-") or "receta"


def stable_slug_for_path(path: Path) -> str:
    rel = path.relative_to(SOURCE_ROOT).as_posix()
    digest = hashlib.sha1(rel.encode("utf-8")).hexdigest()[:8]
    return f"{slugify(path.stem)}-{digest}"


def _read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def _clean_inline_text(value: str) -> str:
    text = re.sub(r"<[^>]+>", " ", str(value or ""), flags=re.IGNORECASE)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _strip_html_text(raw_html: str) -> str:
    no_script = re.sub(r"<script[\s\S]*?</script>", " ", raw_html, flags=re.IGNORECASE)
    no_style = re.sub(r"<style[\s\S]*?</style>", " ", no_script, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", no_style)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _extract_recipe_jsonld(raw_html: str) -> list[dict[str, Any]]:
    blocks = re.findall(
        r"<script[^>]*type=[\\\"']application/ld\+json[\\\"'][^>]*>([\s\S]*?)</script>",
        raw_html,
        flags=re.IGNORECASE,
    )
    recipes: list[dict[str, Any]] = []
    for block in blocks:
        candidate = (block or "").strip()
        if not candidate:
            continue
        try:
            parsed = json.loads(candidate)
        except Exception:
            continue

        stack: list[Any] = [parsed]
        while stack:
            node = stack.pop()
            if isinstance(node, list):
                stack.extend(node)
                continue
            if not isinstance(node, dict):
                continue

            node_type = str(node.get("@type") or "").lower()
            if node_type == "recipe":
                recipes.append(node)
                continue

            graph = node.get("@graph")
            if isinstance(graph, list):
                stack.extend(graph)
    return recipes


def _extract_jsonld_steps(value: Any) -> list[str]:
    if isinstance(value, str):
        text = _clean_inline_text(value)
        return [text] if text else []

    if not isinstance(value, list):
        return []

    steps: list[str] = []
    for item in value:
        if isinstance(item, str):
            text = _clean_inline_text(item)
            if text:
                steps.append(text)
            continue
        if isinstance(item, dict):
            text = _clean_inline_text(item.get("text") or item.get("name") or "")
            if text:
                steps.append(text)
    return steps


def _extract_jsonld_ingredients(value: Any) -> list[str]:
    if isinstance(value, list):
        return [_clean_inline_text(v) for v in value if _clean_inline_text(v)]
    if isinstance(value, str):
        return [_clean_inline_text(value)] if _clean_inline_text(value) else []
    return []


def _best_recipe_jsonld_node(nodes: list[dict[str, Any]]) -> dict[str, Any] | None:
    if not nodes:
        return None

    def _score(node: dict[str, Any]) -> tuple[int, int, int]:
        ingredients = _extract_jsonld_ingredients(node.get("recipeIngredient"))
        steps = _extract_jsonld_steps(node.get("recipeInstructions"))
        has_name = 1 if _clean_inline_text(node.get("name") or "") else 0
        return (len(ingredients), len(steps), has_name)

    return max(nodes, key=_score)


def _compact_jsonld_recipe(node: dict[str, Any]) -> dict[str, Any]:
    return {
        "name": _clean_inline_text(node.get("name") or ""),
        "description": _clean_inline_text(node.get("description") or ""),
        "prepTime": _clean_inline_text(node.get("prepTime") or ""),
        "cookTime": _clean_inline_text(node.get("cookTime") or ""),
        "totalTime": _clean_inline_text(node.get("totalTime") or ""),
        "recipeYield": _clean_inline_text(node.get("recipeYield") or ""),
        "recipeCategory": node.get("recipeCategory"),
        "keywords": _clean_inline_text(node.get("keywords") or ""),
        "recipeIngredient": _extract_jsonld_ingredients(node.get("recipeIngredient")),
        "recipeInstructions": _extract_jsonld_steps(node.get("recipeInstructions")),
    }


def _extract_docx_text(path: Path) -> str:
    with zipfile.ZipFile(path) as zf:
        xml = zf.read("word/document.xml").decode("utf-8", errors="ignore")
    xml = xml.replace("</w:p>", "\n")
    text = re.sub(r"<[^>]+>", " ", xml)
    text = html.unescape(text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()


def _extract_doc_text_via_libreoffice(path: Path) -> str:
    with tempfile.TemporaryDirectory(prefix="doc-to-txt-") as td:
        outdir = Path(td)
        cmd = [
            "libreoffice",
            "--headless",
            "--convert-to",
            "txt:Text",
            "--outdir",
            str(outdir),
            str(path),
        ]
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        txt_files = sorted(outdir.glob("*.txt"))
        if not txt_files:
            return ""
        return txt_files[0].read_text(encoding="utf-8", errors="ignore").strip()


def _extract_pdf_text(path: Path) -> str:
    with tempfile.TemporaryDirectory(prefix="pdf-to-txt-") as td:
        out_txt = Path(td) / "out.txt"
        cmd = ["pdftotext", "-layout", str(path), str(out_txt)]
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if not out_txt.exists():
            return ""
        return out_txt.read_text(encoding="utf-8", errors="ignore").strip()


def extract_source_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".html", ".htm"}:
        raw_html = _read_text_file(path)
        recipe_nodes = _extract_recipe_jsonld(raw_html)
        best = _best_recipe_jsonld_node(recipe_nodes)
        if best is not None:
            compact = _compact_jsonld_recipe(best)
            return "RECIPE_JSONLD_PRIMARY:\n" + json.dumps(compact, ensure_ascii=False, indent=2)

        # fallback only when JSON-LD is missing
        page_text = _strip_html_text(raw_html)
        return "PAGE_TEXT:\n" + page_text[:120000]

    if suffix == ".docx":
        return _extract_docx_text(path)
    if suffix == ".doc":
        return _extract_doc_text_via_libreoffice(path)
    if suffix == ".pdf":
        return _extract_pdf_text(path)
    if suffix == ".txt":
        return _read_text_file(path)
    return ""


In [3]:
def _image_bytes_from_data_url(value: str) -> bytes | None:
    match = re.match(r"^data:image/[^;]+;base64,(.+)$", value.strip(), flags=re.IGNORECASE)
    if not match:
        return None
    payload = re.sub(r"\s+", "", match.group(1))
    try:
        return base64.b64decode(payload, validate=True)
    except Exception:
        return None


def _download_image(url: str, timeout: int = 20) -> bytes | None:
    try:
        response = requests.get(url, timeout=timeout)
        if response.status_code != 200:
            return None
        if not response.content:
            return None
        return response.content
    except Exception:
        return None


def _first_docx_image(path: Path) -> bytes | None:
    try:
        with zipfile.ZipFile(path) as zf:
            media = sorted(
                name
                for name in zf.namelist()
                if name.lower().startswith("word/media/") and not name.endswith("/")
            )
            if not media:
                return None
            return zf.read(media[0])
    except Exception:
        return None


def _first_pdf_image(path: Path) -> bytes | None:
    with tempfile.TemporaryDirectory(prefix="pdf-images-") as td:
        prefix = str(Path(td) / "img")
        cmd = ["pdfimages", "-png", str(path), prefix]
        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        except Exception:
            return None
        pngs = sorted(Path(td).glob("img-*.png"))
        if not pngs:
            return None
        return pngs[0].read_bytes()


def _first_doc_image_via_libreoffice(path: Path) -> bytes | None:
    with tempfile.TemporaryDirectory(prefix="doc-images-") as td:
        outdir = Path(td)
        cmd = [
            "libreoffice",
            "--headless",
            "--convert-to",
            "pdf",
            "--outdir",
            str(outdir),
            str(path),
        ]
        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        except Exception:
            return None
        pdfs = sorted(outdir.glob("*.pdf"))
        if not pdfs:
            return None
        return _first_pdf_image(pdfs[0])


def _html_image_candidates(raw_html: str) -> list[str]:
    candidates: list[str] = []

    for value in re.findall(
        r"<meta[^>]+property=[\\\"']og:image[\\\"'][^>]+content=[\\\"']([^\\\"']+)[\\\"']",
        raw_html,
        flags=re.IGNORECASE,
    ):
        candidates.append(value.strip())

    for node in _extract_recipe_jsonld(raw_html):
        image_field = node.get("image")
        if isinstance(image_field, str):
            candidates.append(image_field.strip())
        elif isinstance(image_field, list):
            for item in image_field:
                if isinstance(item, str):
                    candidates.append(item.strip())

    for value in re.findall(
        r"<img[^>]+src=[\\\"']([^\\\"']+)[\\\"']",
        raw_html,
        flags=re.IGNORECASE,
    ):
        candidates.append(value.strip())

    unique: list[str] = []
    seen: set[str] = set()
    for value in candidates:
        if not value:
            continue
        if value in seen:
            continue
        seen.add(value)
        unique.append(value)
    return unique


def _load_image_candidate(candidate: str, base_dir: Path) -> bytes | None:
    raw = str(candidate or "").strip()
    if not raw:
        return None

    data_url_bytes = _image_bytes_from_data_url(raw)
    if data_url_bytes:
        return data_url_bytes

    parsed = urlparse(raw)
    if parsed.scheme in {"http", "https"}:
        return _download_image(raw)

    cleaned = unquote(raw.split("?", 1)[0].split("#", 1)[0]).strip()
    if not cleaned:
        return None
    local_path = (base_dir / cleaned).resolve() if not cleaned.startswith("/") else Path(cleaned)
    if local_path.exists() and local_path.is_file():
        try:
            return local_path.read_bytes()
        except Exception:
            return None
    return None


def _sibling_image(path: Path) -> bytes | None:
    stem = path.stem.lower()
    for ext in IMAGE_EXTENSIONS:
        candidate = path.with_suffix(ext)
        if candidate.exists() and candidate.is_file():
            return candidate.read_bytes()
    for item in sorted(path.parent.glob("*")):
        if not item.is_file():
            continue
        if item.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        if stem in item.stem.lower():
            return item.read_bytes()
    return None


def extract_image_bytes(path: Path) -> tuple[bytes | None, str]:
    suffix = path.suffix.lower()
    if suffix in IMAGE_EXTENSIONS:
        return path.read_bytes(), "source-image"

    if suffix in {".html", ".htm"}:
        raw_html = _read_text_file(path)
        for candidate in _html_image_candidates(raw_html):
            data = _load_image_candidate(candidate, path.parent)
            if data:
                return data, candidate

    if suffix == ".docx":
        data = _first_docx_image(path)
        if data:
            return data, "docx-embedded"

    if suffix == ".pdf":
        data = _first_pdf_image(path)
        if data:
            return data, "pdf-extracted"

    if suffix == ".doc":
        data = _first_doc_image_via_libreoffice(path)
        if data:
            return data, "doc-converted-pdf"

    sibling = _sibling_image(path)
    if sibling:
        return sibling, "sibling-image"

    return None, ""


def save_card_image_360x270(image_bytes: bytes, slug: str) -> tuple[str, str]:
    image_name = f"{slug}-{int(time.time())}.png"
    out_path = OUTPUT_IMAGE_DIR / image_name

    with Image.open(io.BytesIO(image_bytes)) as img:
        rgb = img.convert("RGB")
        resampling = getattr(Image, "Resampling", Image).LANCZOS
        fitted = ImageOps.fit(rgb, (360, 270), method=resampling)
        stats = ImageStat.Stat(fitted)
        mean = stats.mean[:3]
        fitted.save(out_path, format="PNG", optimize=True)

    color = f"rgb({int(round(mean[0]))}, {int(round(mean[1]))}, {int(round(mean[2]))})"
    route = f"/images/recipes/{image_name}"
    return route, color


def _looks_like_empty_recipe_payload(payload: dict[str, Any]) -> bool:
    if not isinstance(payload, dict):
        return True
    name = str(payload.get("Name") or "").strip().lower()
    ingredients = payload.get("Ingredients")
    steps = payload.get("Steps")

    has_ingredients = False
    if isinstance(ingredients, dict):
        for raw_key, raw_value in ingredients.items():
            key_text = str(raw_key or "").strip()
            value_text = str(raw_value or "").strip()
            if key_text or value_text:
                has_ingredients = True
                break

    has_steps = False
    if isinstance(steps, list):
        for raw_step in steps:
            if str(raw_step or "").strip():
                has_steps = True
                break

    # A title/tags-only payload is still effectively empty for recipe import purposes.
    if not has_ingredients and not has_steps:
        return True

    return (name in {"", "receta"}) and (not has_ingredients) and (not has_steps)
def _parse_iso_duration(duration_text: str) -> str:
    text = str(duration_text or "").strip().upper()
    if not text.startswith("P"):
        return str(duration_text or "").strip()

    day_match = re.search(r"(\d+)D", text)
    h_match = re.search(r"(\d+)H", text)
    m_match = re.search(r"(\d+)M", text)

    days = int(day_match.group(1)) if day_match else 0
    hours = int(h_match.group(1)) if h_match else 0
    minutes = int(m_match.group(1)) if m_match else 0

    total_hours = days * 24 + hours
    parts: list[str] = []
    if total_hours:
        parts.append(f"{total_hours} h")
    if minutes:
        parts.append(f"{minutes} min")
    return " ".join(parts) if parts else "No especificado"


def _parse_persons(value: Any) -> str:
    text = _clean_inline_text(value or "")
    if not text:
        return "No especificado"
    match = re.search(r"\d+", text)
    return match.group(0) if match else text


def _split_amount_ingredient(raw: str) -> tuple[str, str]:
    text = _clean_inline_text(raw)
    if not text:
        return "", ""
    match = re.match(
        r"^([~≈]?[\d]+(?:[\.,]\d+)?(?:\s*[/-]\s*[\d]+(?:[\.,]\d+)?)?\s*(?:g|kg|ml|l|tsp|tbsp|cup|cups|oz|lb|ud|uds|unid(?:ad(?:es)?)?)?)(?:\s+)(.+)$",
        text,
        flags=re.IGNORECASE,
    )
    if match:
        return match.group(2).strip(), match.group(1).strip()
    return text, ""


def _fallback_payload_from_html_jsonld(path: Path) -> dict[str, Any] | None:
    if path.suffix.lower() not in {".html", ".htm"}:
        return None

    raw_html = _read_text_file(path)
    best = _best_recipe_jsonld_node(_extract_recipe_jsonld(raw_html))
    if best is None:
        return None

    ingredients: dict[str, str] = {}
    for row in _extract_jsonld_ingredients(best.get("recipeIngredient")):
        name, amount = _split_amount_ingredient(row)
        if not name:
            continue
        key = name
        idx = 2
        while key in ingredients:
            key = f"{name} ({idx})"
            idx += 1
        ingredients[key] = amount

    steps = _extract_jsonld_steps(best.get("recipeInstructions"))

    categories = best.get("recipeCategory")
    if isinstance(categories, list):
        tags = [_clean_inline_text(v).lower() for v in categories if _clean_inline_text(v)]
    else:
        cat_text = _clean_inline_text(categories or "")
        tags = [cat_text.lower()] if cat_text else []

    keywords = _clean_inline_text(best.get("keywords") or "")
    if keywords:
        tags.extend([chunk.strip().lower() for chunk in keywords.split(",") if chunk.strip()])

    dedup_tags: list[str] = []
    seen: set[str] = set()
    for tag in tags:
        if tag and tag not in seen:
            seen.add(tag)
            dedup_tags.append(tag)

    return {
        "Name": _clean_inline_text(best.get("name") or "") or "Receta",
        "Ingredients": ingredients,
        "Steps": steps,
        "Preparation time": _parse_iso_duration(best.get("prepTime") or "") or "No especificado",
        "Total time": _parse_iso_duration(best.get("totalTime") or "") or "No especificado",
        "Nºpersonas": _parse_persons(best.get("recipeYield") or ""),
        "Tags": dedup_tags,
        "Tools": [],
    }


def build_llm_input(path: Path, extracted_text: str, max_chars: int = 60000) -> str:
    rel = path.relative_to(SOURCE_ROOT)
    excerpt = (extracted_text or "").strip()
    if len(excerpt) > max_chars:
        excerpt = excerpt[:max_chars]
    return (
        f"SOURCE_PATH: {rel}\n"
        f"SOURCE_TYPE: {path.suffix.lower()}\n\n"
        f"Extract recipe details and normalize for the webapp JSON schema.\n"
        f"Output language required: Spanish (es-ES).\n"
        f"TEXT:\n{excerpt}"
    )


def write_recipe_json(slug: str, payload: dict) -> Path:
    out_path = OUTPUT_JSON_DIR / f"{slug}.json"
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return out_path


def convert_recipe_file(path: Path, *, model: str = MODEL) -> tuple[dict, str]:
    source_text = extract_source_text(path)
    if not source_text.strip():
        raise ValueError("No textual content extracted")

    llm_input = build_llm_input(path, source_text)
    recipe_payload = recipe_text_to_webapp_json(
        llm_input,
        model=model,
        force_english=(OUTPUT_LANGUAGE == "en"),
        default_card_color=DEFAULT_CARD_COLOR,
    )

    # Retry/fallback path for HTML sources when the model returns an empty/default payload.
    if _looks_like_empty_recipe_payload(recipe_payload) and path.suffix.lower() in {".html", ".htm"}:
        fallback = _fallback_payload_from_html_jsonld(path)
        if fallback is not None:
            retry_input = build_llm_input(
                path,
                "STRUCTURED_RECIPE_FALLBACK:\n" + json.dumps(fallback, ensure_ascii=False, indent=2),
                max_chars=20000,
            )
            retry_payload = recipe_text_to_webapp_json(
                retry_input,
                model=model,
                force_english=(OUTPUT_LANGUAGE == "en"),
                default_card_color=DEFAULT_CARD_COLOR,
            )
            if not _looks_like_empty_recipe_payload(retry_payload):
                recipe_payload = retry_payload
            else:
                recipe_payload = fallback

    if _looks_like_empty_recipe_payload(recipe_payload):
        raise ValueError("Empty/default recipe payload after extraction/LLM fallback")

    image_bytes, image_source = extract_image_bytes(path)
    if image_bytes:
        image_route, _ = save_card_image_360x270(image_bytes, stable_slug_for_path(path))
        recipe_payload["card image file"] = image_route
    else:
        recipe_payload.pop("card image file", None)

    return recipe_payload, image_source


def discover_source_files() -> list[Path]:
    files = [
        path
        for path in SOURCE_ROOT.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_SOURCE_EXTENSIONS
    ]
    files.sort(key=lambda p: str(p).lower())
    return files


def run_batch(
    *,
    max_files: int | None = None,
    skip_existing: bool = True,
    sleep_seconds: float = 0.0,
    model: str = MODEL,
) -> list[dict[str, Any]]:
    files = discover_source_files()
    if max_files is not None:
        files = files[:max_files]

    rows: list[dict[str, Any]] = []

    for path in tqdm(files, desc="Converting recipes"):
        slug = stable_slug_for_path(path)
        out_json = OUTPUT_JSON_DIR / f"{slug}.json"

        if skip_existing and out_json.exists():
            rows.append(
                {
                    "source": str(path.relative_to(SOURCE_ROOT)),
                    "slug": slug,
                    "status": "skipped",
                    "json_path": str(out_json.relative_to(PROJECT_ROOT)),
                    "image_source": "",
                    "error": "",
                }
            )
            continue

        try:
            payload, image_source = convert_recipe_file(path, model=model)
            if _looks_like_empty_recipe_payload(payload):
                raise ValueError("Empty/default recipe payload; JSON not saved")
            written = write_recipe_json(slug, payload)
            rows.append(
                {
                    "source": str(path.relative_to(SOURCE_ROOT)),
                    "slug": slug,
                    "status": "ok",
                    "json_path": str(written.relative_to(PROJECT_ROOT)),
                    "image_source": image_source,
                    "error": "",
                }
            )
        except Exception as exc:
            rows.append(
                {
                    "source": str(path.relative_to(SOURCE_ROOT)),
                    "slug": slug,
                    "status": "error",
                    "json_path": "",
                    "image_source": "",
                    "error": f"{exc.__class__.__name__}: {exc}",
                }
            )

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    report_path = OUTPUT_JSON_DIR / "_run_report.json"
    report_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"Run report written to: {report_path}")
    return rows


In [4]:
# Patch overrides: robust DOC/DOCX/PDF extraction + Gemini file retry + safer image handling
from src.recipe_importer import (
    _extract_source_text as _shared_extract_source_text,
    _extract_image_bytes as _shared_extract_image_bytes,
    _fallback_payload_from_plain_text as _shared_fallback_payload_from_plain_text,
    _guess_mime_type as _shared_guess_mime_type,
)
from src.LLMs_funcs import recipe_file_to_webapp_json


def extract_source_text(path: Path) -> str:
    """Use the shared importer extractor (handles RTF .doc and more DOCX variants)."""
    return _shared_extract_source_text(path)


def extract_image_bytes(path: Path) -> tuple[bytes | None, str]:
    """Use shared image extraction so unsupported WMF/EMF docx media are skipped."""
    return _shared_extract_image_bytes(path)


def convert_recipe_file(path: Path, *, model: str = MODEL) -> tuple[dict, str]:
    source_text = extract_source_text(path)
    suffix = path.suffix.lower()

    recipe_payload = None
    if source_text.strip():
        llm_input = build_llm_input(path, source_text)
        recipe_payload = recipe_text_to_webapp_json(
            llm_input,
            model=model,
            force_english=(OUTPUT_LANGUAGE == "en"),
            default_card_color=DEFAULT_CARD_COLOR,
        )
    elif suffix in {".pdf", ".doc", ".docx"}:
        # Scanned/embedded-content documents may have no local text; ask Gemini to read the file directly.
        recipe_payload = recipe_file_to_webapp_json(
            path.read_bytes(),
            _shared_guess_mime_type(path),
            context_text="",
            model=model,
            force_english=(OUTPUT_LANGUAGE == "en"),
            default_card_color=DEFAULT_CARD_COLOR,
        )
    else:
        raise ValueError("No textual content extracted")

    # Retry/fallback path for HTML sources when the model returns an empty/default payload.
    if _looks_like_empty_recipe_payload(recipe_payload) and suffix in {".html", ".htm"}:
        fallback = _fallback_payload_from_html_jsonld(path)
        if fallback is not None:
            retry_input = build_llm_input(
                path,
                "STRUCTURED_RECIPE_FALLBACK:\n" + json.dumps(fallback, ensure_ascii=False, indent=2),
                max_chars=20000,
            )
            retry_payload = recipe_text_to_webapp_json(
                retry_input,
                model=model,
                force_english=(OUTPUT_LANGUAGE == "en"),
                default_card_color=DEFAULT_CARD_COLOR,
            )
            if not _looks_like_empty_recipe_payload(retry_payload):
                recipe_payload = retry_payload
            else:
                recipe_payload = fallback

    # Retry with Gemini file-reading for office docs/PDFs when text-based parsing returned an empty/default recipe.
    if _looks_like_empty_recipe_payload(recipe_payload) and suffix in {".pdf", ".doc", ".docx"}:
        try:
            context_bits = []
            if source_text.strip():
                context_bits.append(f"LOCAL_EXTRACTED_TEXT:\n{source_text}")
            retry_file_payload = recipe_file_to_webapp_json(
                path.read_bytes(),
                _shared_guess_mime_type(path),
                context_text="\n\n".join(context_bits),
                model=model,
                force_english=(OUTPUT_LANGUAGE == "en"),
                default_card_color=DEFAULT_CARD_COLOR,
            )
            if not _looks_like_empty_recipe_payload(retry_file_payload):
                recipe_payload = retry_file_payload
        except Exception as exc:  # noqa: BLE001
            print(f"[WARN] Gemini file retry failed for {path}: {exc}")

    # Deterministic fallback for office/plain-text documents when the LLM still returns the default empty payload.
    if _looks_like_empty_recipe_payload(recipe_payload):
        fallback_plain = _shared_fallback_payload_from_plain_text(path, source_text)
        if fallback_plain is not None:
            recipe_payload = fallback_plain

    if _looks_like_empty_recipe_payload(recipe_payload):
        raise ValueError("Empty/default recipe payload after extraction/LLM fallback")

    image_bytes, image_source = extract_image_bytes(path)
    if image_bytes:
        try:
            image_route, _ = save_card_image_360x270(image_bytes, stable_slug_for_path(path))
            recipe_payload["card image file"] = image_route
        except Exception as exc:  # noqa: BLE001
            print(f"[WARN] Skipping unsupported image for {path}: {exc}")
            recipe_payload.pop("card image file", None)
            image_source = ""
    else:
        recipe_payload.pop("card image file", None)

    return recipe_payload, image_source


In [5]:
files = discover_source_files()
print(f"Total supported files: {len(files)}")
files[:10]


Total supported files: 7767


[PosixPath('/home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Chocolate_Ingles/$250 chocolate chunk cookies__r89701.html'),
 PosixPath('/home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Chocolate_Ingles/Almond addiction__r89698.html'),
 PosixPath('/home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Chocolate_Ingles/Almond and Matcha Cake__r133922.html'),
 PosixPath('/home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Chocolate_Ingles/Almond biscotti__r90145.html'),
 PosixPath('/home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Chocolate_Ingles/Almond briouats with honey and sesame seeds __r165554.html'),
 PosixPath('/home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Chocolate_Ingles/Almond Briouats with Honey and Sesame Seeds __r170308.html'),
 PosixPath('/home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Chocolate_Ingles/Almond Butter Oatmeal Cookies__r508464.html'),
 PosixPath('/home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Chocolate_Ingles/Almond Cake wit

In [6]:
# Dry run on a single file to inspect output format.
sample = discover_source_files()[0]
sample_payload, sample_image_source = convert_recipe_file(sample)
print("Sample source:", sample.relative_to(SOURCE_ROOT))
print("Image source:", sample_image_source or "none")
print(json.dumps(sample_payload, ensure_ascii=False, indent=2)[:2000])


Sample source: Chocolate_Ingles/$250 chocolate chunk cookies__r89701.html
Image source: https://assets.tmecosys.com/image/upload/t_web767x639/img/recipe/ras/Assets/042C0610-E26A-4768-A9A4-52F814035BD3/Derivates/69521228-C6EF-4CBE-8372-69B5D2FF9027.jpg
{
  "Name": "$250 Galletas con trozos de chocolate",
  "Ingredients": {
    "avena en hojuelas": "300 g",
    "almendras tostadas": "150 g",
    "nueces de macadamia tostadas y saladas": "150 g",
    "chocolate con leche": "250 g",
    "azúcar moreno": "150 g",
    "azúcar blanco": "220 g",
    "mantequilla": "250 g",
    "extracto de vainilla natural": "1 cucharada",
    "huevos": "2",
    "harina común": "250 g",
    "bicarbonato de sodio": "1 cucharadita",
    "levadura en polvo": "1 cucharadita",
    "sal marina": "1 pizca",
    "gotas de chocolate": "250 g"
  },
  "Steps": [
    "Coloque la avena en el bol mezclador y pique 3 seg/velocidad 6. Transfiera a un bol y reserve.",
    "Coloque las almendras y las macadamias en el bol mezcl

In [10]:
# Recommended first batch for validation.
rows = run_batch(max_files=None, skip_existing=True, sleep_seconds=0.15, model=MODEL)

ok = sum(1 for row in rows if row["status"] == "ok")
skipped = sum(1 for row in rows if row["status"] == "skipped")
errors = [row for row in rows if row["status"] == "error"]

print(f"ok={ok} skipped={skipped} errors={len(errors)}")
if errors:
    print("First errors:")
    for row in errors[:5]:
        print("-", row["source"], "->", row["error"])


Converting recipes:  88%|████████▊ | 6838/7767 [00:00<00:00, 12945.39it/s]

[WARN] Gemini file retry failed for /home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Indice.docx: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Unsupported MIME type: application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'status': 'INVALID_ARGUMENT'}}


Converting recipes:  88%|████████▊ | 6838/7767 [00:20<00:00, 12945.39it/s]Model gemini-2.5-flash timed out after 40.0s; retrying (1/2)
Model gemini-2.5-flash timed out after 40.0s; retrying (2/2)
Model gemini-2.5-flash failed after 2 retries; trying fallback model gemini-2.5-flash-lite
Converting recipes: 100%|█████████▉| 7744/7767 [02:49<00:01, 19.78it/s]   

[WARN] Gemini file retry failed for /home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Recetas/8. bebidas/A Queimada.docx: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Unsupported MIME type: application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'status': 'INVALID_ARGUMENT'}}
[WARN] Gemini file retry failed for /home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Recetas/Diccionario.docx: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Unsupported MIME type: application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'status': 'INVALID_ARGUMENT'}}
[WARN] Gemini file retry failed for /home/rafael/E/Python_Projects/Recetas_webapp/Recetas/Recetas/Nuevas o corregidas.docx: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Unsupported MIME type: application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'status': 'INVALID_ARGUMENT'}}


Converting recipes: 100%|██████████| 7767/7767 [02:58<00:00, 43.43it/s]

Run report written to: /home/rafael/E/Python_Projects/Recetas_webapp/data/recipes_generated/_run_report.json
ok=5 skipped=7760 errors=2
First errors:
- Recetas/4. Pescado/Lubina al estilo turco.pdf -> ValueError: Empty/default recipe payload after extraction/LLM fallback
- Recetas/7. Postres/Gofres.pdf -> ValueError: Empty/default recipe payload after extraction/LLM fallback


In [11]:
[x["source"] for x in errors]

['Recetas/4. Pescado/Lubina al estilo turco.pdf',
 'Recetas/7. Postres/Gofres.pdf']

In [12]:
# Delete previously generated empty/default recipes so they can be regenerated.
# Default is dry-run (preview only). Set DRY_RUN = False to actually delete.
DRY_RUN = False
DELETE_LINKED_IMAGES = False  # Set True to also delete /images/recipes/... referenced by deleted JSONs.

def find_empty_generated_recipe_jsons(json_dir: Path = OUTPUT_JSON_DIR) -> list[Path]:
    files = sorted(path for path in json_dir.glob('*.json') if path.name != '_run_report.json')
    empty_files: list[Path] = []
    for file_path in files:
        try:
            payload = json.loads(file_path.read_text(encoding='utf-8'))
        except Exception:
            # Invalid JSON should be reviewed manually; do not auto-delete here.
            continue
        if isinstance(payload, dict) and _looks_like_empty_recipe_payload(payload):
            empty_files.append(file_path)
    return empty_files

def _linked_local_image_path(payload: dict[str, Any]) -> Path | None:
    route = str(payload.get('card image file') or '').strip()
    if not route or not route.startswith('/images/recipes/'):
        return None
    local_path = PROJECT_ROOT / route.lstrip('/')
    return local_path

def delete_empty_generated_recipe_jsons(*, dry_run: bool = True, delete_linked_images: bool = False) -> list[dict[str, str]]:
    targets = find_empty_generated_recipe_jsons()
    deleted_rows: list[dict[str, str]] = []

    for json_path in targets:
        payload = json.loads(json_path.read_text(encoding='utf-8'))
        image_path = _linked_local_image_path(payload) if isinstance(payload, dict) else None
        row = {
            'json': str(json_path.relative_to(PROJECT_ROOT)),
            'image': str(image_path.relative_to(PROJECT_ROOT)) if image_path and image_path.exists() else '',
        }
        deleted_rows.append(row)

        if dry_run:
            continue

        json_path.unlink(missing_ok=True)
        if delete_linked_images and image_path is not None and image_path.exists():
            image_path.unlink(missing_ok=True)

    return deleted_rows

rows_deleted = delete_empty_generated_recipe_jsons(
    dry_run=DRY_RUN,
    delete_linked_images=DELETE_LINKED_IMAGES,
)

print(f"Empty/default generated JSONs found: {len(rows_deleted)}")
action = 'Would delete' if DRY_RUN else 'Deleted'
for row in rows_deleted[:20]:
    print(f"- {action}: {row['json']}")
    if row['image']:
        image_action = 'Would also delete image' if DRY_RUN else 'Deleted image'
        print(f"  {image_action}: {row['image']}")
if len(rows_deleted) > 20:
    print(f"... and {len(rows_deleted) - 20} more")


Empty/default generated JSONs found: 0


In [ ]:
# Full run (uncomment when ready)
# rows = run_batch(max_files=None, skip_existing=True, sleep_seconds=0.2, model=MODEL)


In [14]:
# Add source-based tags to generated recipe JSONs.
#
# Rules:
# - Files from Recetas/Chocolate_Ingles and Recetas/España -> add tag: thermomix
# - Files from Recetas/Recetas and Recetas/Recetas-libros -> add tag: mama
#
# Uses discover_source_files() + stable_slug_for_path() to find the generated JSON for each source.
TAG_PATCH_DRY_RUN = False  # Preview only. Set to False to write changes.

import unicodedata


def _tag_patch_norm(value: object) -> str:
    text = str(value or '').strip()
    if not text:
        return ''
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text.casefold()


def _source_extra_tags_for_patch(rel_path: str) -> list[str]:
    rel = str(rel_path or '').replace('\\', '/').lstrip('./')
    rel_norm = _tag_patch_norm(rel)

    extra: list[str] = []

    thermomix_prefixes = (
        _tag_patch_norm('Chocolate_Ingles/') ,
        _tag_patch_norm('España/'),
        _tag_patch_norm('Recetas/Chocolate_Ingles/'),
        _tag_patch_norm('Recetas/España/'),
    )
    mama_prefixes = (
        _tag_patch_norm('Recetas/'),
        _tag_patch_norm('Recetas-libros/'),
        _tag_patch_norm('Recetas/Recetas/'),
        _tag_patch_norm('Recetas/Recetas-libros/'),
    )

    if any(rel_norm.startswith(prefix) for prefix in thermomix_prefixes):
        extra.append('thermomix')
    if any(rel_norm.startswith(prefix) for prefix in mama_prefixes):
        extra.append('mama')

    return extra


def add_source_based_tags_to_generated_jsons(*, dry_run: bool = True) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []

    for source_path in discover_source_files():
        rel_source = str(source_path.relative_to(SOURCE_ROOT)).replace('\\', '/')
        extra_tags = _source_extra_tags_for_patch(rel_source)
        if not extra_tags:
            continue

        slug = stable_slug_for_path(source_path)
        json_path = OUTPUT_JSON_DIR / f'{slug}.json'
        if not json_path.exists():
            continue

        try:
            payload = json.loads(json_path.read_text(encoding='utf-8'))
        except Exception:
            continue
        if not isinstance(payload, dict):
            continue

        existing_tags_raw = payload.get('Tags')
        if isinstance(existing_tags_raw, list):
            tags = [str(tag).strip() for tag in existing_tags_raw if str(tag).strip()]
        else:
            tags = []

        seen = {_tag_patch_norm(tag) for tag in tags if _tag_patch_norm(tag)}
        added: list[str] = []
        for tag in extra_tags:
            norm_tag = _tag_patch_norm(tag)
            if not norm_tag or norm_tag in seen:
                continue
            tags.append(tag)
            seen.add(norm_tag)
            added.append(tag)

        if not added:
            continue

        payload['Tags'] = tags
        rows.append(
            {
                'source': rel_source,
                'json': str(json_path.relative_to(PROJECT_ROOT)),
                'added_tags': ', '.join(added),
            }
        )

        if not dry_run:
            json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

    return rows


rows_tag_patch = add_source_based_tags_to_generated_jsons(dry_run=TAG_PATCH_DRY_RUN)
print(f'Recipes to update: {len(rows_tag_patch)}')
print('Mode:', 'DRY_RUN (preview only)' if TAG_PATCH_DRY_RUN else 'WRITE')
for row in rows_tag_patch[:30]:
    action = 'Would update' if TAG_PATCH_DRY_RUN else 'Updated'
    print(f"- {action}: {row['json']}  (+{row['added_tags']})")
    print(f"  source: {row['source']}")
if len(rows_tag_patch) > 30:
    print(f"... and {len(rows_tag_patch) - 30} more")


Recipes to update: 7676
Mode: WRITE
- Updated: data/recipes_generated/250-chocolate-chunk-cookies-r89701-b88bd210.json  (+thermomix)
  source: Chocolate_Ingles/$250 chocolate chunk cookies__r89701.html
- Updated: data/recipes_generated/almond-addiction-r89698-3ecb8e17.json  (+thermomix)
  source: Chocolate_Ingles/Almond addiction__r89698.html
- Updated: data/recipes_generated/almond-and-matcha-cake-r133922-fe8e13b6.json  (+thermomix)
  source: Chocolate_Ingles/Almond and Matcha Cake__r133922.html
- Updated: data/recipes_generated/almond-biscotti-r90145-eca4d11d.json  (+thermomix)
  source: Chocolate_Ingles/Almond biscotti__r90145.html
- Updated: data/recipes_generated/almond-briouats-with-honey-and-sesame-seeds-r165554-c211b9dd.json  (+thermomix)
  source: Chocolate_Ingles/Almond briouats with honey and sesame seeds __r165554.html
- Updated: data/recipes_generated/almond-briouats-with-honey-and-sesame-seeds-r170308-8c7ef839.json  (+thermomix)
  source: Chocolate_Ingles/Almond Briouats 